In [1]:
import os
from pathlib import Path
import pandas as pd

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)

    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p

    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path (the one containing data/raw): ").strip().strip('"')

    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    """Try OUT_DIR first; if the OS rejects the write, fall back to the home folder."""
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part2a_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
districts = pd.read_csv(RAW_DIR / "districts.csv")
age_sex = pd.read_csv(RAW_DIR / "age_sex.csv")

proj_district = "D05"
d_row = districts[districts["district_id"] == proj_district].iloc[0]

p_2020 = float(d_row["population_2020"])
p_2025 = float(age_sex[age_sex["district_id"] == proj_district]["population_2025"].sum())

# 1 Jan 2020 -> 1 Jan 2025 is 5 years; 1 Jan 2025 -> 1 Jan 2030 is 5 years
b_annual = (p_2025 - p_2020) / 5.0
p_2030_arith = p_2025 + 5.0 * b_annual

g_annual = (p_2025 / p_2020) ** (1.0 / 5.0)
r_annual = g_annual - 1.0
p_2030_geom = p_2025 * (g_annual ** 5.0)

proj_2a = pd.DataFrame([{
    "district_id": proj_district,
    "pop_2020_estimate": p_2020,
    "pop_2025_estimate": p_2025,
    "arith_annual_increment_estimate": b_annual,
    "geom_annual_factor_estimate": g_annual,
    "geom_annual_rate_pct_estimate": r_annual * 100.0,
    "pop_2030_arith_conditional": p_2030_arith,
    "pop_2030_geom_conditional": p_2030_geom,
    "diff_geom_minus_arith": p_2030_geom - p_2030_arith,
}])

# Show the table first, then save
display(proj_2a)
safe_save(lambda p: proj_2a.to_csv(p, index=False), "part2a_arith_geom_d05.csv")

BASE_DIR: C:\Users\SOOQ ELASER\Documents\GitHub\Group-4-Assignment-2-Two-


,district_id,pop_2020_estimate,pop_2025_estimate,arith_annual_increment_estimate,geom_annual_factor_estimate,geom_annual_rate_pct_estimate,pop_2030_arith_conditional,pop_2030_geom_conditional,diff_geom_minus_arith
0,D05,137858.0,152444.0,2917.2,1.020318,2.031829,167030.0,168573.26478,1543.26478


Saved: C:\Users\SOOQ ELASER\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part2a_arith_geom_d05.csv
